In [6]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import traceback
from typing import List, Dict, Any

import pandas as pd
from tqdm.auto import tqdm

In [2]:
# =========================
# Config
# =========================

# lyricwhiz repo
LYRICWHIZ_REPO = Path("/home/hbli/songformer/repo/LyricWhiz")
# audio path
AUDIO_DIR = Path("/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios")
# output path
RUN_ROOT = Path("/mnt/ssd/hbli/songformer/runs/lyricwhiz_eda")
# temp input path
INPUT_DIR = RUN_ROOT / "input_wavs"
WHISPER_OUTPUT_DIR = RUN_ROOT / "whisper_results"
# line-level output path
FINAL_JSON_DIR = RUN_ROOT / "linelevel_outputs"

# 2 case
TARGET_FILES = [
    # "HX_0003_6foot7foot.wav",
    # "HX_0008_america.wav",
]
MAX_FILES = 2

# params
WHISPER_MODEL = "large"
WHISPER_PROMPT = "lyrics: "
WHISPER_LANGUAGE = "en"
VOCAL_THRESHOLD = 0.00

INPUT_DIR.mkdir(parents=True, exist_ok=True)
WHISPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_JSON_DIR.mkdir(parents=True, exist_ok=True)

print("LYRICWHIZ_REPO:", LYRICWHIZ_REPO)
print("AUDIO_DIR:", AUDIO_DIR)
print("RUN_ROOT:", RUN_ROOT)

LYRICWHIZ_REPO: /home/hbli/songformer/repo/LyricWhiz
AUDIO_DIR: /mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios
RUN_ROOT: /mnt/ssd/hbli/songformer/runs/lyricwhiz_eda


In [3]:
def get_audio_files(audio_dir: Path, target_files: List[str], max_files: int) -> List[Path]:
    wavs = sorted(audio_dir.glob("*.wav"))
    if target_files:
        wanted = set(target_files)
        wavs = [p for p in wavs if p.name in wanted]
    if max_files is not None:
        wavs = wavs[:max_files]
    return wavs

def safe_symlink_or_copy(src: Path, dst: Path):
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

audio_files = get_audio_files(AUDIO_DIR, TARGET_FILES, MAX_FILES)

print(f"Selected {len(audio_files)} files:")
for p in audio_files:
    print(" -", p.name)

# 清空并重建输入目录
for old in INPUT_DIR.glob("*"):
    if old.is_symlink() or old.is_file():
        old.unlink()
    elif old.is_dir():
        shutil.rmtree(old)

for p in audio_files:
    safe_symlink_or_copy(p, INPUT_DIR / p.name)

print("\nPrepared INPUT_DIR:")
for p in sorted(INPUT_DIR.glob("*")):
    print(" -", p.name)

Selected 2 files:
 - HX_0003_6foot7foot.wav
 - HX_0006_aint2proud2beg.wav

Prepared INPUT_DIR:
 - HX_0003_6foot7foot.wav
 - HX_0006_aint2proud2beg.wav


In [7]:
def run_cmd(cmd, cwd=None):
    print("Running:\n", " ".join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
    )
    print("\nSTDOUT:\n", result.stdout[:5000])
    print("\nSTDERR:\n", result.stderr[:5000])

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}")
    return result


cmd = [
    sys.executable,
    str(LYRICWHIZ_REPO / "code" / "run_whisper.py"),
    "--model", WHISPER_MODEL,
    "--prompt", WHISPER_PROMPT,
    "--language", WHISPER_LANGUAGE,
    "--input_dir", str(INPUT_DIR),
    "--output_dir", str(WHISPER_OUTPUT_DIR),
    "--n_shard", "1",
    "--shard_rank", "0",
    "--threshold", str(VOCAL_THRESHOLD),
]

run_cmd(cmd, cwd=LYRICWHIZ_REPO)

Running:
 /home/hbli/songformer/env/miniforge3/envs/lyricwhiz/bin/python /home/hbli/songformer/repo/LyricWhiz/code/run_whisper.py --model large --prompt lyrics:  --language en --input_dir /mnt/ssd/hbli/songformer/runs/lyricwhiz_eda/input_wavs --output_dir /mnt/ssd/hbli/songformer/runs/lyricwhiz_eda/whisper_results --n_shard 1 --shard_rank 0 --threshold 0.0

STDOUT:
 

STDERR:
 
  0%|                                              | 0.00/2.88G [00:00<?, ?iB/s]
  0%|                                    | 48.0k/2.88G [00:00<2:07:46, 403kiB/s]
  0%|                                     | 160k/2.88G [00:00<1:04:00, 804kiB/s]
  0%|                                      | 456k/2.88G [00:00<29:01, 1.77MiB/s]
  0%|                                     | 1.17M/2.88G [00:00<12:48, 4.01MiB/s]
  0%|                                     | 3.05M/2.88G [00:00<05:23, 9.55MiB/s]
  0%|                                     | 7.73M/2.88G [00:00<02:14, 22.8MiB/s]
  0%|▏                                    | 12.7M/2.

RuntimeError: Command failed with code 1

In [ ]:
rows = []
for wav in audio_files:
    json_path = WHISPER_OUTPUT_DIR / f"{wav.name}.json"
    rows.append(
        {
            "audio": wav.name,
            "json_exists": json_path.exists(),
            "json_path": str(json_path),
        }
    )

pd.DataFrame(rows)